# Week 3: Perform Exploratory Data Analysis

**Project:** Airline Passenger Demand Forecasting using BTS T-100 Domestic Segment (All Carriers)

**Week 3 Objectives**
- Split model-eligible observations into chronological training, validation, and test partitions.
- Conduct exploratory data analysis **only on training partition**.
- Identify data-quality and modeling issues.
- Document preprocessing recommendations for subsequent modeling weeks.

> Project predicts future passenger demand, so observations are split by ordered **forecast month** to keep future periods out of training set and preserves test period as true unseen holdout.

## ===== WEEK 3 START =====


## 1. Environment and reusable project code

Week 3 builds on validated Week 2 ingestion pipeline stored in `Src/week2_src_t100_ingest_explore.py`. 
Week 3 logic is stored in `Src/week3_src_eda_split.py`.

This cell:
- Finds project root automatically.
- Imports Week 3 module.
- Confirms paths used for generated tables and figures.


In [ ]:
from pathlib import Path
from IPython.display import display, Image
import importlib.util
import pandas as pd

def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for path in [start, *start.parents]:
        if (path / "Data").exists() and (path / "Src").exists() and (path / "Notebooks").exists():
            return path
    raise FileNotFoundError("Could not locate the project root.")

PROJECT_ROOT = find_project_root()
REPORTS_DIR = PROJECT_ROOT / "Reports"
WEEK3_REPORTS_DIR = REPORTS_DIR / "week3"
WEEK3_TABLES_DIR = WEEK3_REPORTS_DIR / "week3_tables"
WEEK3_FIGURES_DIR = WEEK3_REPORTS_DIR / "week3_figures"
SRC_DIR = PROJECT_ROOT / "Src"

module_path = SRC_DIR / "week3_src_eda_split.py"
spec = importlib.util.spec_from_file_location("week3_eda", module_path)
week3 = importlib.util.module_from_spec(spec)
spec.loader.exec_module(week3)

print("Project root:", PROJECT_ROOT)
print("Week 3 source:", module_path)
print("Reports directory:", REPORTS_DIR)


## 2. Rebuild Week 2 Analytical Table

Week 2 pipeline:
- Reads annual BTS ZIP archives
- Filters project scope to scheduled passenger service
- Aggregates to **carrier × origin × destination × month** level
- Constructs next-month passenger target


In [ ]:
monthly = week3.load_week2_monthly(PROJECT_ROOT)

print(f"Carrier-route-month rows: {len(monthly):,}")
print(f"Columns: {monthly.shape[1]:,}")
print(f"Reference period: {monthly['Period'].min():%Y-%m} through {monthly['Period'].max():%Y-%m}")
print(f"Rows with observed next-month target: {monthly[week3.TARGET].notna().sum():,}")
print(f"Rows without observed next-month target: {monthly[week3.TARGET].isna().sum():,}")


## 3. Determine Model Eligibility Before Splitting

Rows without observed `TargetPassengersNextMonth` cannot be used for supervised regression because outcome is unknown. They are **excluded rather than imputed**.


In [ ]:
model_data, eligibility_summary = week3.prepare_model_data(monthly)
display(eligibility_summary)


## 4. Chronological 70% / 15% / 15% Partition

Split is based on ordered unique **forecast months**:

- **Training:** earliest 70% of forecast months; used for EDA and later model fitting.
- **Validation:** next 15%; reserved for later model comparison/tuning.
- **Test:** most recent 15% (subject to whole-month rounding); untouched until final model evaluation.

All rows belonging to same forecast month remain in same partition. 
This prevents calendar month from being represented simultaneously in training / future holdout.


In [ ]:
(
    train_df,
    validation_df,
    test_df,
    split_calendar,
    split_summary,
) = week3.chronological_split(model_data)

display(split_summary)

print("Chronological integrity checks:")
print("  Training ends before validation starts:",
      train_df[week3.TARGET_TIME_COLUMN].max() < validation_df[week3.TARGET_TIME_COLUMN].min())
print("  Validation ends before test starts:",
      validation_df[week3.TARGET_TIME_COLUMN].max() < test_df[week3.TARGET_TIME_COLUMN].min())
print("  All eligible rows assigned:",
      len(train_df) + len(validation_df) + len(test_df) == len(model_data))


## 5. Training-only EDA

From this point forward, **no distributional EDA is conducted on validation or test observations**. The following analysis uses `train_df` only:

- structural/profile checks
- missingness and data types
- logical-value and continuity checks
- descriptive statistics, skewness, and kurtosis
- target distribution
- numeric predictor correlations
- calendar seasonality
- monthly time trend
- categorical cardinality

The validation and test partitions remain reserved for later modeling weeks.


In [ ]:
(
    training_profile,
    missingness,
    data_issues,
    numeric_summary,
    target_correlations,
    seasonality,
    monthly_trend,
    categorical_profile,
    key_findings,
) = week3.build_training_eda(monthly, train_df)

display(training_profile)


### 5.1 Missingness, cardinality, and data types

In [ ]:
display(missingness)

### 5.2 Data-quality and continuity checks


In [ ]:
display(data_issues)

### 5.3 Numeric descriptive statistics

In [ ]:
display(numeric_summary)

### 5.4 Numeric relationships with the next-month target

In [ ]:
display(target_correlations)

### 5.5 Calendar seasonality

In [ ]:
display(seasonality)

### 5.6 Categorical cardinality

In [ ]:
display(categorical_profile)

## 6. Save Week 3 artifacts and visualize the training results

The saved figures below use the training partition only, except the split-size chart, which simply documents partition sizes.


In [ ]:
results = week3.Week3Results(
    monthly=monthly,
    model_data=model_data,
    train_df=train_df,
    validation_df=validation_df,
    test_df=test_df,
    split_calendar=split_calendar,
    split_summary=split_summary,
    eligibility_summary=eligibility_summary,
    training_profile=training_profile,
    missingness=missingness,
    data_issues=data_issues,
    numeric_summary=numeric_summary,
    target_correlations=target_correlations,
    seasonality=seasonality,
    monthly_trend=monthly_trend,
    categorical_profile=categorical_profile,
    key_findings=key_findings,
    preprocessing_recommendations=week3._preprocessing_recommendations(),
)

week3.save_outputs(results, PROJECT_ROOT)

print("Saved tables:", WEEK3_TABLES_DIR)
print("Saved figures:", WEEK3_FIGURES_DIR)


In [ ]:
figure_paths = [
    WEEK3_FIGURES_DIR / "week3_figure_split_sizes.png",
    WEEK3_FIGURES_DIR / "week3_figure_training_target_distribution.png",
    WEEK3_FIGURES_DIR / "week3_figure_training_demand_over_time.png",
    WEEK3_FIGURES_DIR / "week3_figure_training_seasonality.png",
    WEEK3_FIGURES_DIR / "week3_figure_training_target_correlations.png",
]

for path in figure_paths:
    if path.exists():
        print(path.name)
        display(Image(filename=str(path)))


## 7. Key findings and preprocessing recommendations


In [ ]:
display(key_findings)
display(results.preprocessing_recommendations)


## 8. Week 3 completion checks

The analysis is considered complete when:
- every eligible observation belongs to exactly one chronological partition,
- validation/test months occur strictly after the training months,
- EDA tables/figures are based on training data only,
- the notebook runs from top to bottom without errors,
- report-ready tables/figures exist.


In [ ]:
# Confirm strict chronological separation among the partitions.
assert (
    train_df[week3.TARGET_TIME_COLUMN].max()
    < validation_df[week3.TARGET_TIME_COLUMN].min()
), "Training and validation periods overlap."

assert (
    validation_df[week3.TARGET_TIME_COLUMN].max()
    < test_df[week3.TARGET_TIME_COLUMN].min()
), "Validation and test periods overlap."

# Confirm every modeling observation belongs to exactly one partition.
assert (
    len(train_df) + len(validation_df) + len(test_df)
    == len(model_data)
), "Partition row counts do not sum to the modeling dataset."

# Confirm critical Week 3 outputs were generated.
assert (
    WEEK3_TABLES_DIR / "week3_results_summary.json"
).exists(), "Week 3 results summary was not generated."

assert (
    WEEK3_FIGURES_DIR / "week3_figure_training_target_distribution.png"
).exists(), "Week 3 target-distribution figure was not generated."

print("All Week 3 checks passed.")
print("===== WEEK 3 END =====")